# SeQUeNCe in 3 Minutes

In this introductory three-minute tutorial, we will briefly cover the necessary basics of SeQUeNCe. We will request entanglement pairs between two quantum routers, Alice and Bob. The goal of this tutorial is to gain a basic usage of SeQUeNCe.

The whole tutorial can additionally be found at <a href="https://github.com/sequence-toolbox/SeQUeNCe/blob/master/docs/source/tutorial/sequence3min/three_minute.py" target="_blank" rel="noopener">three_minute.py</a>, and is broken down into 5 steps, discussed in detail in the following sections.


### Step 1: Generate the Two-Node Topology

Before building the simulation, we first need a network configuration file that defines the two nodes and the channels between them. For this tutorial, we will use SeQUeNCe's built-in topology generator, which is implemented in <a href="https://github.com/sequence-toolbox/SeQUeNCe/blob/master/sequence/utils/config_generator_cli.py" target="_blank" rel="noopener">config_generator_cli.py</a>.
The `generate-topology` command-line executable is installed automatically when SeQUeNCe is installed, so it is available from the same Python environment without any additional setup.

```bash
generate-topology linear 2 --memory-size 1 --output two_node.json
```

The generate-topology command has 5 main arguments here:
- `linear` specifies a linear network, with all quantum routers in a line
- `2` specifies that we want 2 nodes in the network
- `--memory-size 1` specifies that each quantum router has 1 quantum memory
- `--output two_node.json` specifies that we want to save the network topology to the file `two_node.json`
- `--directory .` specifies the output directory (we'll use the current directory here)

In [10]:
!generate-topology linear 2 --memory-size 1 --output two_node.json --directory .

### Step 2: Import the Required Modules

We begin by importing the request application, the router-network topology class, and the entanglement-generation protocol controls. The `RequestApp` class provides a simple application interface for making an entanglement request between two routers.


In [4]:
from sequence.app.request_app import RequestApp
from sequence.topology.router_net_topo import RouterNetTopo
from sequence.constants import SINGLE_HERALDED, SECOND
from sequence.entanglement_management.generation import EntanglementGenerationA, EntanglementGenerationB

This tutorial uses the single-heralded entanglement-generation protocol. We set both the router-side and BSM-side generation protocols to `SINGLE_HERALDED` before loading the network.

In [5]:
EntanglementGenerationA.set_global_type(SINGLE_HERALDED)
EntanglementGenerationB.set_global_type(SINGLE_HERALDED)

### Step 3: Load the Network

Next, we load the generated JSON file into a `RouterNetTopo`. This constructs the timeline, quantum routers, BSM node, and communication channels defined in the topology file.

In [11]:
network_topo = RouterNetTopo(config_source="two_node.json")
tl = network_topo.get_timeline()

### Step 4: Attach the Application Module to the Nodes

Loop over the node objects, and then attach the application to the nodes and make the entanglement request.

In [12]:
name_to_app = {}
for router in network_topo.get_nodes_by_type(RouterNetTopo.QUANTUM_ROUTER):
    name_to_app[router.name] = RequestApp(router)

### Step 5: Run the Simulation

Finally, we initialize the timeline, start Alice's request to Bob, and run the simulation.
The request has the following arguments:
- `responder` is the node that we want to generate entanglement with. In this case, we're choosing the only other router in the network.
- `start_t` is the simulation time at which the request will begin being filled.
- `end_t` is the simulation time at which the request will stop being filled.
- `memo_size` is the number of memories to reserve for attempting entnalgmenet generation. In this case, we set it equal to 1, as there is only one memory per router.
- `fidelity` is the required fidelity of the entanglement pairs that will be generated. Any pairs generated with a lower fidelity will be purified before delivery to the application.

In [13]:
tl.init()
alice = "router_0"
bob = "router_1"
name_to_app[alice].start(responder=bob, start_t=1 * SECOND, end_t=2.5 * SECOND, memo_size=1, fidelity=0.8)
tl.run()

After the timeline finishes running, we can check the number of entangled pairs between Alice and Bob and the throughput.

In [14]:
print(f"Entangled pair count between Alice and Bob: {name_to_app[alice].memory_counter}")
print(f"The throughput is {name_to_app[alice].get_throughput()} pairs per second")

Entangled pair count between Alice and Bob: 105
The throughput is 70.0 pairs per second
